In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from models.topic import TopicAnalyzer
from scipy.stats import linregress
from scipy.signal import detrend, periodogram
from statsmodels.graphics.tsaplots import plot_acf

topic_analyzer = TopicAnalyzer.load("models/bertopic_model")

df = pd.read_csv("data/results.csv")

In [ ]:
sentiment_by_year = (
    df.groupby("year")["sentiment"]
    .mean()
)

print(sentiment_by_year.max())
print(sentiment_by_year.min())

rolling_mean = sentiment_by_year.rolling(
    window=5,
    center=True
).mean()

df["decade"] = (df["year"] // 10) * 10

decade_summary = (
    df["decade"]
    .value_counts()
    .sort_index()
    .rename_axis("decade")
    .reset_index(name="count")
)

decade_summary["percentage"] = (
    decade_summary["count"] / decade_summary["count"].sum() * 100
).round(2)

# print(decade_summary)

In [ ]:
sentiment_cols = [
    "sentiment_negative",
    "sentiment_neutral",
    "sentiment_positive"
]

df["sentiment_class"] = (
    df[sentiment_cols]
    .idxmax(axis=1)
    .str.replace("sentiment_", "")
)

sentiment_share = (
    df["sentiment_class"]
    .value_counts(normalize=True)
    .mul(100)
    .reindex(["negative", "neutral", "positive"])
)

In [ ]:
sentiment_order = ["negative", "neutral", "positive"]

sentiment_counts = (
    df["sentiment_class"]
    .value_counts()
    .reindex(sentiment_order)
)

sentiment_percentage = (
    sentiment_counts / sentiment_counts.sum() * 100
)

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(
    sentiment_counts.index,
    sentiment_counts.values,
    color=["red", "blue", "green"]
)

ax.set_xlabel("Sentyment")
ax.set_ylabel("Liczba utworów")
ax.set_ylim(0, sentiment_counts.max() * 1.1)
ax.set_title("Liczba utworów według dominującego sentymentu")

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["Negatywny", "Neutralny", "Pozytywny"])

ax.set_axisbelow(True)
ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.5
)

ax.bar_label(
    bars,
    labels=[f"{x:.1f}%" for x in sentiment_percentage],
    padding=3
)

plt.show()

In [ ]:
df["sentiment_class"] = df["sentiment"].apply(
    lambda x: "Pozytywny" if x > 0 else "Negatywny"
)

sentiment_counts2 = df["sentiment_class"].value_counts()

sentiment_percentage = (
    sentiment_counts2 / sentiment_counts2.sum() * 100
).round(2)

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(
    sentiment_counts2.index,
    sentiment_counts2.values,
    color=["green", "red"]
)

ax.set_xlabel("Sentyment")
ax.set_ylabel("Liczba utworów")
ax.set_ylim(0, sentiment_counts2.max() * 1.1)
ax.set_title("Liczba utworów według dominującego sentymentu")

ax.set_xticks([0, 1])

ax.set_axisbelow(True)
ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.5
)

ax.bar_label(
    bars,
    labels=[f"{x:.1f}%" for x in sentiment_percentage],
    padding=3
)

plt.show()

In [ ]:
def plot_sentiment_histogram(df):
    plt.figure(figsize=(10, 6))

    sns.histplot(
        data=df,
        x="sentiment",
        bins=20
    )

    plt.axvline(
        0,
        linestyle="--",
        linewidth=1
    )

    plt.xlabel("Wartość sentymentu")
    plt.ylabel("Liczba utworów")
    plt.xlim(-1, 1)

    plt.tight_layout()
    plt.show()

plot_sentiment_histogram(df)

In [ ]:
x = sentiment_by_year.index.to_numpy()
y = sentiment_by_year.to_numpy()

regression = linregress(x, y)

trend = regression.intercept + regression.slope * x

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    sentiment_by_year.index,
    sentiment_by_year,
    label="Średni sentyment"
)

ax.plot(
    rolling_mean.index,
    rolling_mean,
    label="5-letnia średnia krocząca"
)

'''
ax.plot(
    x,
    trend,
    label="Linia trendu"
)
'''

ax.set_ylim(-1, 1)
ax.set_xlabel("Rok")
ax.set_ylabel("Średni sentyment")
ax.legend()

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.5
)

plt.show()

In [ ]:
print("Slope:", regression.slope)
print("Intercept:", regression.intercept)
print("R²:", regression.rvalue ** 2)
print("p-value:", regression.pvalue)

In [ ]:
df["decade"] = (df["year"] // 10) * 10

sns.boxplot(
    data=df,
    x="decade",
    y="sentiment"
).set(
    xlabel="Dekada",
    ylabel="Średni sentyment"
)

In [ ]:
sentiment_detrended = detrend(sentiment_by_year.values)

plot_acf(sentiment_detrended, lags=25, title=None)
plt.xlabel("Opóźnienie [lata]")
plt.ylabel("Autokorelacja")
plt.show()

In [ ]:
frequencies, power = periodogram(sentiment_detrended)

periods = 1 / frequencies[1:]
power = power[1:]

plt.plot(periods, power, marker="o")
plt.xlabel("Okres [lata]")
plt.ylabel("Moc")
plt.xlim(2, 30)
plt.show()

In [ ]:
from scipy.signal import periodogram, find_peaks

signal = sentiment_by_year.values

frequencies, power = periodogram(signal)

frequencies = frequencies[1:]
power = power[1:]

peaks, _ = find_peaks(power)

peak_frequencies = frequencies[peaks]
peak_power = power[peaks]

# okres odpowiadający częstotliwości
peak_periods = 1 / peak_frequencies

import pandas as pd

peaks_df = pd.DataFrame({
    "frequency": peak_frequencies,
    "power": peak_power,
    "period_years": peak_periods
})

peaks_df = peaks_df.sort_values(
    "power",
    ascending=False
)

print(peaks_df)

In [ ]:
yearly_change = sentiment_by_year.diff()

largest_changes = yearly_change.abs().sort_values(ascending=False)

print(largest_changes.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    sentiment_by_year.index,
    sentiment_by_year.values
)

threshold = yearly_change.abs().quantile(0.95)

change_points = yearly_change.abs().nlargest(10)

ax.scatter(
    change_points.index,
    sentiment_by_year.loc[change_points.index],
    zorder=3
)

ax.set_xlabel("Rok")
ax.set_ylabel("Średni sentyment")

plt.show()

In [ ]:
topic_by_decade = pd.crosstab(
    df[df["topic"] != -1]["topic"],
    df[df["topic"] != -1]["decade"],
    normalize="columns"
) * 100

sns.heatmap(
    topic_by_decade,
    annot=True,
    fmt=".1f",
    cmap="Blues"
).set(
    xlabel="Dekada",
    ylabel="Temat"
)

In [ ]:
outliers = df[df["topic"] == -1]
outliers[["artist", "song", "lyrics"]].head(10)

In [ ]:
topic_analyzer.visualize_topics()

In [ ]:
topic_info = topic_analyzer.get_topic_info()
print(topic_info)

In [ ]:
fig = topic_analyzer.model.visualize_barchart()

fig.update_layout(
    title={
        "text": "Słowa reprezentujące najpopularniejsze tematy",
        "x": 0.5,
        "xanchor": "center"
    }
)

fig.show()

In [ ]:
df["year"] = pd.to_numeric(df["year"], errors="coerce")

df_time = df.dropna(subset=["year"]).copy()
df_time["year"] = df_time["year"].astype(int)

topics_over_time = topic_analyzer.model.topics_over_time(
    docs=df_time["lyrics"].tolist(),
    timestamps=df_time["year"].tolist()
)

In [ ]:
fig = topic_analyzer.model.visualize_topics_over_time(
    topics_over_time,
    topics=[0,1,2,3]
)

fig.update_layout(
    title={
            "text": "Najpopularniejsze tematy w czasie",
            "x": 0.5,
            "xanchor": "center"
        }
)

fig.show()

In [ ]:
print(topic_analyzer.__dict__.keys())

In [ ]:
topic_info = topic_analyzer.model.get_topic_info()

for _, row in topic_info.iterrows():
    if row["Topic"] != -1:
        print(f"\n{'='*80}")
        print(f"TOPIC {row['Topic']} ({row['Count']} utworów)")
        print(f"{'='*80}")

        for i, doc in enumerate(row["Representative_Docs"][:10], 1):
            print(f"\n{i}. {doc}")

In [ ]:
from wordcloud import WordCloud

topics = list(range(12))

for topic in topics:
    words = dict(topic_analyzer.model.get_topic(topic))

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color="white"
    ).generate_from_frequencies(words)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Temat {topic}")
    plt.show()